In [ ]:
import os
#AA#> Colab Env Variables
os.environ['MPLBACKEND'] = 'notebook'
HF_api_key = os.environ.get("HF_TOKEN")
os.environ['HF_TOKEN'] = HF_api_key

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ImageRAG
 - https://rotem-shalev.github.io/ImageRAG/

In [ ]:
! pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
mainDir = "/content/drive/MyDrive/RAGArtist"

In [ ]:
import os, urllib.request
import zipfile
with zipfile.ZipFile(f'{mainDir}/ImageRAG-main.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

In [ ]:
%cd /content/ImageRAG-main
! conda env create -f environment.yml

In [ ]:
with zipfile.ZipFile(f'{mainDir}/OmniGen-main.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

### Batch OmniGen Run

In [ ]:
#! unzip /content/1.zip -d /content/ImageRAG-main/fromWeb
! mv /content/ImageRAG-main/fromWeb/7/* /content/ImageRAG-main/fromWeb

In [ ]:
import pandas as pd
import shutil, os, subprocess

In [ ]:
prompts = pd.read_excel(f'{mainDir}/عکس‌نوشت ها_out.xlsx',index_col = 'id')

for id in prompts.index[393:400]:
  row = prompts.loc[id]
  prompt = row['Persian']
  english = row['English']
  print(prompt,'\n',english)

  dest_folder = f"datasets/{row['English']}"
  os.makedirs(dest_folder, exist_ok = True)

  for j in range(5): #len(eval(row['Persian_Concepts']))):
    # Folder of Persian concept images
    source_folder = f"fromWeb/{english}_{j}"

    try:
        shutil.copytree(source_folder, dest_folder,dirs_exist_ok=True)
        print(f"Folder '{source_folder}' copied successfully to '{dest_folder}'.")
    except FileExistsError:
        print(f"Error: Destination folder '{dest_folder}' already exists.")
    except Exception as e:
        print(f"An error occurred: {e}")

  #AA#> Run RAG with OmniGen
  procStr = ['--prompt',  prompt, '--mode', 'generation','--out_name', english,
             '--dataset', english,'--omnigen_path', '/content/OmniGen-main',
             '--openai_api_key',
             '$OPENAI_API_KEY'
          ]

  ragProcess = subprocess.run([ "conda", "run", "-n", "ImageRAG",
                                "python", "imageRAG_OmniGen.py"] + procStr,
                                 stdout=subprocess.PIPE,
                                 stderr=subprocess.PIPE,
                                 text=True # decode bytes → str
                          )

  print("STDOUT:\n", ragProcess.stdout)
  print("STDERR:\n", ragProcess.stderr)

  saveOut = subprocess.run(["cp", "-r", "/content/ImageRAG-main/results", f"{mainDir}/Results/ImageRAG"])
  print(f'Output is saved for image {id}\n')

In [ ]:
%%shell
zip -r /content/results.zip "/content/ImageRAG-main/results"

### Single OmniGen Run

In [ ]:
%%shell
eval "$(conda shell.bash hook)" # copy conda command to shell
conda activate ImageRAG

python imageRAG_OmniGen.py \
--prompt "مرد جوان با پای شکسته، در حال هدایت پهپاد برای فیلمبرداری از قله ی وارونه لای دره " \
--mode "generation" \
--out_name  "Ariadaad2" \
--dataset "Ariadaad2" \
--omnigen_path /content/OmniGen-main \
--openai_api_key "$OPENAI_API_KEY"

conda deactivate

In [ ]:
! rm /content/ImageRAG-main/results/*

In [ ]:
torch.cuda.empty_cache()

In [ ]:
مجسمه شاعر در سمت راست، یک کوزه بزرگ در چپ، و ربات انسان نما در وسط، در پارک آب و آتش
  یک زن مهندس در چپ، یک ربات انسان نما در راست، و دو خودروی برقی در فضای باز تخت جمشید
یک دانشجوی دختر در حال آموزش هوش مصنوعی به دو طوطی، در میدان نقش جهان
یک دختر با لباس قاجاری در حال اسکیت‌سواری در سمت راست خیابان ولیعصر تهران، یک خودروی برقی در سمت چپ
دختر با شال قرمز نشسته در کافه مدرن، پشت سرش کاشی‌کاری سنتی
پیرزن روستایی با چادر گلدار در فروشگاه زنجیره‌ای
زن با مانتوی سفید در حال گرفتن سلفی جلوی تخت جمشید

### Single SDXL Run

In [ ]:
%%shell
eval "$(conda shell.bash hook)" # copy conda command to shell
conda activate ImageRAG

python imageRAG_SDXL.py \
--prompt "کلاس درس هوش مصنوعی برای گربه ها در میدان نقش جهان اصفهان، سه بالن در آسمان " \
--mode "generation" \
--out_name 'Panda_AI_Tabriz2' \
--dataset 'Panda_AI_Tabriz2' \
--openai_api_key "$OPENAI_API_KEY"

conda deactivate

conda deactivate

## Web Search

In [ ]:
# DuckDuckGo
! pip install ddgs

# GenArtist
 - READ: [Virt Env in Colab](https://stackoverflow.com/questions/48512569/is-it-possible-to-activate-virtualenv-in-google-colab-bin-sh-1-source-not)

In [ ]:
%cd /content

In [ ]:
%%bash
pip install virtualenv
virtualenv genArtistEnv

In [ ]:
! source /content/genArtistEnv/bin/activate

In [ ]:
import os, urllib.request
import zipfile
with zipfile.ZipFile('GenArtist-main.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/genArtistEnv/bin')

In [ ]:
# Making necessary changes in the scripts

mainDir = "/content/genArtistEnv/bin/GenArtist-main"
textUpdate = {
              'from kornia import SamplePadding': 'from kornia.constants import SamplePadding',
              'map_location="cpu"':'map_location="cuda"'
              }

for origText in textUpdate:
  newText = textUpdate[origText]
  pyFiles = []

  for root, dirs, files in os.walk(mainDir):
    for file in files:
      if file.endswith('.py'):
        if origText in str(open(os.path.join(root, file),'r').read()):
          pyFiles.append(os.path.join(root, file))

  print(f"{len(pyFiles)} with string {origText}")

  for file in pyFiles:
    orig = open(file,'r').read()
    if  origText in str(orig):
      f = open(file,'w')
      udapte = orig.replace(origText, newText)
      f.write(udapte)
      f.close()
      print(f'{file} is updated with text {newText}')


In [ ]:
%cd /content/genArtistEnv/bin/GenArtist-main/
!source /content/genArtistEnv/bin/activate; pip install pip==23.3

In [ ]:
! source /content/genArtistEnv/bin/activate; pip install -r requirements.txt

In [ ]:
#AA#> Colab Virtual Env requirements
%%bash
source /content/genArtistEnv/bin/activate;
pip install httpx==0.27.2;
pip install groundingdino-py;
pip install matplotlib;
pip install gradio==3.40.0;
pip install pydantic==1.10.21;
pip install k-diffusion;
pip install taming-transformers-rom1504;
pip install ipython;
pip install pyrallis;

In [ ]:
# Pre-trained Models

# Inputs
mainDir = "/content/genArtistEnv/bin/GenArtist-main"
hf = "https://huggingface.co"
preTrMls = {
             f"{hf}/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth?download=true":f"{mainDir}/GroundingDINO/weights/groundingdino_swint_ogc.pth",
             f"{hf}/camenduru/AnyDoor/resolve/main/dinov2_vitg14_pretrain.pth?download=true":f"{mainDir}/AnyDoor/path/dinov2_vitg14_pretrain.pth",
             f"{hf}/spaces/xichenhku/AnyDoor/resolve/main/epoch%3D1-step%3D8687.ckpt?download=true":f"{mainDir}/AnyDoor/path/epoch=1-step=8687.ckpt", #16.8 GB
             f"{hf}/spaces/InpaintAI/Inpaint-Anything/resolve/main/pretrained_models/big-lama/config.yaml?download=true": f"{mainDir}/Inpaint-Anything/pretrained_models/big-lama/config.yaml",
             f"{hf}/osunlp/InstructPix2Pix-MagicBrush/resolve/main/MagicBrush-epoch-000168.ckpt?download=true": f"{mainDir}/InstructPix2Pix/checkpoints/MagicBrush-epoch-000168.ckpt", #14.8 GB
             f"{hf}/spaces/InpaintAI/Inpaint-Anything/resolve/main/pretrained_models/big-lama/models/best.ckpt?download=true": f"{mainDir}/Inpaint-Anything/pretrained_models/big-lama/models/best.ckpt",

          }

# prepare progressbar
import progressbar
pbar = None

def show_progress(block_num, block_size, total_size):
    global pbar
    if pbar is None:
        pbar = progressbar.ProgressBar(maxval=total_size)
        pbar.start()

    downloaded = block_num * block_size
    if downloaded < total_size:
        pbar.update(downloaded)
    else:
        pbar.finish()
        pbar = None

# Downloading Models
for hfModel in list(preTrMls.keys()):
  os.makedirs(preTrMls[hfModel][:-4], exist_ok=True)
  urllib.request.urlretrieve(hfModel, preTrMls[hfModel],show_progress)
  print(f"Downloaded {hfModel} to {preTrMls[hfModel]}")

In [ ]:
ragPaths = [f'inputs/image_{ind}.png' for ind in [1,2,3]]

'"' + '" ,"'.join(ragPaths) + '"'

In [ ]:
%cd /content/genArtistEnv/bin/GenArtist-main/
!source /content/genArtistEnv/bin/activate; python demo_t2i.py

In [ ]:
! rm /content/genArtistEnv/bin/GenArtist-main/inputs/*